In [14]:
import pandas as pd
import sqlite3
import numpy as np

In [15]:
con = sqlite3.connect("PLStats.db")

In [46]:
stats = pd.read_excel("data/PremierLeagueStats.xlsx", sheet_name="Player Standard Stats", decimal=',')
stats.to_sql(name="stats", con=con, if_exists="replace")
positions = pd.read_excel("data/PremierLeagueStats.xlsx", sheet_name="Player Positions")
positions.to_sql(name="positions", con=con, if_exists="replace", index=True, index_label="pos_id")

617

In [17]:
pd.read_sql_query("SELECT * FROM stats", con=con)

,index,Player ID,Player Name,Player Surname,Nation Code,Squad ID,Age,Born,Matches Played,Starts,...,Assists/90,G+A/90,G-PK/90,G+A-PK/90,xG/90,xAG/90,xG+xAG/90,npxG/90,npxG+xAG/90,Player Fullname
0,0,1,Brenden,Aaronson,USA,11,25-081,2000,20,14,...,0.22,0.51,0.29,0.51,0.21,0.20,0.41,0.21,0.41,Brenden Aaronson
1,1,2,Zach,Abbott,ENG,16,19-243,2006,1,0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Zach Abbott
2,2,3,Joshua,Acheampong,ENG,7,19-251,2006,8,6,...,0.00,0.19,0.19,0.19,0.12,0.01,0.13,0.12,0.13,Joshua Acheampong
3,3,4,Tyler,Adams,USA,3,26-331,1999,15,15,...,0.08,0.23,0.15,0.23,0.03,0.01,0.04,0.03,0.04,Tyler Adams
4,4,5,Tosin,Adarabioyo,ENG,7,28-109,1997,10,6,...,0.00,0.00,0.00,0.00,0.01,0.09,0.11,0.01,0.11,Tosin Adarabioyo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,488,489,Leny,Yoro,FRA,14,20-059,2005,21,12,...,0.00,0.00,0.00,0.00,0.05,0.00,0.06,0.05,0.06,Leny Yoro
489,489,490,Oleksandr,Zinchenko,UKR,16,29-027,1996,5,4,...,0.00,0.00,0.00,0.00,0.02,0.13,0.15,0.02,0.15,Oleksandr Zinchenko
490,490,491,Joshua,Zirkzee,NED,14,24-234,2001,14,4,...,0.18,0.55,0.37,0.55,0.36,0.17,0.52,0.36,0.52,Joshua Zirkzee
491,491,492,Martín,Zubimendi,ESP,1,26-343,1999,21,20,...,0.05,0.20,0.15,0.20,0.10,0.05,0.15,0.10,0.15,Martín Zubimendi


In [18]:
con.execute('ALTER TABLE stats RENAME COLUMN "G-PK/90" TO np_goals_per90')
con.execute('ALTER TABLE stats RENAME COLUMN "npxG/90" TO npxg_per90')
con.execute('ALTER TABLE stats RENAME COLUMN "Minutes" TO minutes')

In [19]:
con.execute('ALTER TABLE stats RENAME COLUMN "Squad ID" TO squad_id')

In [20]:
con.execute('ALTER TABLE stats RENAME COLUMN "Player Fullname" TO player_name')

In [21]:
con.execute('ALTER TABLE stats RENAME COLUMN "Player ID" TO player_id')

In [22]:
con.execute('ALTER TABLE positions RENAME COLUMN "Player ID" TO player_id')

In [23]:
con.execute('ALTER TABLE positions RENAME COLUMN "Player Position" TO position')

In [24]:
pd.read_sql_query("""
    SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes
    FROM stats
    WHERE minutes > 300
    ORDER BY g_xg_diff ASC LIMIT 15
""", con=con)

,player_name,g_xg_diff,minutes
0,Justin Devenny,-0.32,513
1,David Brooks,-0.31,937
2,Andrey Santos,-0.29,520
3,Beto,-0.28,835
4,Anthony Gordon,-0.25,998
5,Liam Delap,-0.22,575
6,Jean-Philippe Mateta,-0.20,1730
7,Estêvão Willian,-0.20,610
8,Jhon Arias,-0.19,1048
9,Niclas Füllkrug,-0.19,409


In [25]:
pd.read_sql_query("""
    SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes
    FROM stats
    WHERE minutes > 300
    ORDER BY g_xg_diff DESC LIMIT 15
""", con=con)

,player_name,g_xg_diff,minutes
0,Eli Junior Kroupi,0.54,576
1,Mathys Tel,0.45,470
2,Mateus Mane,0.29,452
3,Emi Buendía,0.26,769
4,Samuel Chukwueze,0.22,398
5,Bruno Guimarães,0.20,1748
6,Antoine Semenyo,0.19,1798
7,Harry Wilson,0.19,1496
8,Zian Flemming,0.18,667
9,Morgan Rogers,0.18,1867


In [26]:
pd.read_sql_query("""
    WITH xg_perform AS (
        SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes
        FROM stats
        WHERE minutes > 300)
    SELECT player_name, g_xg_diff, minutes FROM xg_perform
    ORDER BY g_xg_diff ASC LIMIT 15
""", con=con)

,player_name,g_xg_diff,minutes
0,Justin Devenny,-0.32,513
1,David Brooks,-0.31,937
2,Andrey Santos,-0.29,520
3,Beto,-0.28,835
4,Anthony Gordon,-0.25,998
5,Liam Delap,-0.22,575
6,Jean-Philippe Mateta,-0.20,1730
7,Estêvão Willian,-0.20,610
8,Jhon Arias,-0.19,1048
9,Niclas Füllkrug,-0.19,409


In [27]:
pd.read_sql_query("""
    WITH xg_perform AS (
        SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes, squad_id
        FROM stats
        WHERE minutes > 300)
    SELECT player_name, g_xg_diff, minutes, squad_id,
           RANK() OVER (PARTITION BY squad_id ORDER BY g_xg_diff DESC) AS Ranking
    FROM xg_perform
""", con=con)

,player_name,g_xg_diff,minutes,squad_id,Ranking
0,Eberechi Eze,0.17,923,1,1
1,Gabriel Magalhães,0.10,1266,1,2
2,Mikel Merino,0.09,940,1,3
3,Leandro Trossard,0.08,1131,1,4
4,Declan Rice,0.08,1696,1,5
...,...,...,...,...,...
367,Tolu Arokodare,-0.07,673,20,18
368,Yerson Mosquera,-0.07,866,20,18
369,Marshall Munetsi,-0.07,875,20,18
370,Jørgen Strand Larsen,-0.08,1313,20,21


In [28]:
pd.read_sql_query("""
    SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes, squad_id, AVG(np_goals_per90 - npxg_per90) OVER () FROM stats
""", con=con)

,player_name,g_xg_diff,minutes,squad_id,AVG(np_goals_per90 - npxg_per90) OVER ()
0,Brenden Aaronson,0.08,1245,11,-0.011542
1,Zach Abbott,0.00,45,16,-0.011542
2,Joshua Acheampong,0.07,470,7,-0.011542
3,Tyler Adams,0.12,1195,3,-0.011542
4,Tosin Adarabioyo,-0.01,651,7,-0.011542
...,...,...,...,...,...
488,Leny Yoro,-0.05,1141,14,-0.011542
489,Oleksandr Zinchenko,-0.02,353,16,-0.011542
490,Joshua Zirkzee,0.01,490,14,-0.011542
491,Martín Zubimendi,0.05,1769,1,-0.011542


In [29]:
pd.read_sql_query("""
    WITH xg_perform AS (
        SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes, squad_id
        FROM stats
        WHERE minutes > 300),
        rank_table AS (
            SELECT player_name, g_xg_diff, minutes, squad_id,
            RANK() OVER (PARTITION BY squad_id ORDER BY g_xg_diff DESC) AS ranking
            FROM xg_perform)
    SELECT player_name, g_xg_diff, minutes, squad_id FROM rank_table
        WHERE ranking = 1    
""", con=con)

,player_name,g_xg_diff,minutes,squad_id
0,Eberechi Eze,0.17,923,1
1,Emi Buendía,0.26,769,2
2,Eli Junior Kroupi,0.54,576,3
3,Thiago,0.17,1752,4
4,Danny Welbeck,0.17,1121,5
5,Zian Flemming,0.18,667,6
6,Moisés Caicedo,0.14,1342,7
7,Daniel Muñoz,0.08,1259,8
8,Kiernan Dewsbury-Hall,0.18,1257,9
9,Samuel Chukwueze,0.22,398,10


In [30]:
pd.read_sql_query(sql="""SELECT typeof(minutes), typeof(np_goals_per90), typeof(npxg_per90) FROM stats LIMIT 1""", con=con)

,typeof(minutes),typeof(np_goals_per90),typeof(npxg_per90)
0,integer,real,real


In [31]:
pd.read_sql_query(sql = """WITH avg_min_g AS (
    SELECT np_goals_per90 - npxg_per90 AS g_xg_diff, minutes,
    AVG(np_goals_per90 - npxg_per90) OVER () AS avg_diff,
    AVG(minutes) OVER () AS avg_min 
    FROM stats
    WHERE minutes > 300), 
    pwr_cov AS (
    SELECT POWER(g_xg_diff - avg_diff, 2) AS pwr_diff_xg,
    POWER(minutes - avg_min, 2) AS pwr_diff_min,
    (g_xg_diff - avg_diff)*(minutes - avg_min) AS cov
    FROM avg_min_g), 
    elements AS (
    SELECT 1.0/(COUNT(*)-1)*SUM(pwr_diff_xg) AS var_xg, 1.0/(COUNT(*)-1)*SUM(pwr_diff_min) AS var_min,
    1.0/(COUNT(*)-1)*SUM(cov) AS cov FROM pwr_cov)
    SELECT cov/(SQRT(var_min)*SQRT(var_xg)) AS corr_pearson FROM elements""", con = con)

,corr_pearson
0,0.103409


In [32]:
stats["g_xg_diff"] = stats["G-PK/90"] - stats["npxG/90"]
stats[stats["Minutes"] > 300][["Minutes","g_xg_diff"]].corr(method="pearson")

,Minutes,g_xg_diff
Minutes,1.000000,0.103409
g_xg_diff,0.103409,1.000000


In [33]:
pd.read_sql_query("SELECT * FROM positions", con=con)

,pos_id,player_id,position
0,0,1,FW
1,1,1,MF
2,2,2,DF
3,3,3,DF
4,4,4,MF
...,...,...,...
612,612,490,DF
613,613,491,FW
614,614,491,MF
615,615,492,MF


In [34]:
pd.read_sql_query("SELECT COUNT(*) FROM stats WHERE minutes > 300", con=con)

,COUNT(*)
0,372


In [35]:
pd.read_sql_query("""SELECT * FROM positions WHERE "Player Position" <> "GK" """, con=con)

,pos_id,player_id,position
0,0,1,FW
1,1,1,MF
2,2,2,DF
3,3,3,DF
4,4,4,MF
...,...,...,...
612,612,490,DF
613,613,491,FW
614,614,491,MF
615,615,492,MF


In [36]:
pd.read_sql_query("""SELECT player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, minutes, squad_id, positions.position
                        FROM stats
                        INNER JOIN positions
                        ON stats.player_id = positions.player_id
                        WHERE minutes > 300 AND positions.position <> 'GK'
                        """, con=con)

,player_name,g_xg_diff,minutes,squad_id,position
0,Brenden Aaronson,0.08,1245,11,FW
1,Brenden Aaronson,0.08,1245,11,MF
2,Joshua Acheampong,0.07,470,7,DF
3,Tyler Adams,0.12,1195,3,MF
4,Tosin Adarabioyo,-0.01,651,7,DF
...,...,...,...,...,...
431,Oleksandr Zinchenko,-0.02,353,16,DF
432,Joshua Zirkzee,0.01,490,14,FW
433,Joshua Zirkzee,0.01,490,14,MF
434,Martín Zubimendi,0.05,1769,1,MF


In [37]:
pd.read_sql_query("""SELECT COUNT(positions.player_id) - COUNT(DISTINCT positions.player_id)
                        FROM stats
                        INNER JOIN positions
                        ON stats.player_id = positions.player_id
                        WHERE minutes > 300 AND positions.position <> 'GK'
                        """, con=con)

,COUNT(positions.player_id) - COUNT(DISTINCT positions.player_id)
0,92


In [38]:
pd.read_sql_query("""WITH players AS (
                                      SELECT stats.player_id AS player_id, player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, squad_id, positions.position AS position, 
                                      positions.pos_id, RANK() OVER (PARTITION BY positions.player_id ORDER BY pos_id) AS prim_pos, minutes
                                      FROM stats
                                      INNER JOIN positions
                                      ON stats.player_id = positions.player_id),
                    more_than_2 AS (
                                    SELECT player_name, g_xg_diff, position, squad_id, COUNT(player_id) OVER (PARTITION BY squad_id, position) AS same_pos_squad, minutes, prim_pos
                                    FROM players
                                    WHERE minutes > 300 AND position <> 'GK' AND prim_pos = 1)
                    SELECT player_name, g_xg_diff, position, squad_id,
                    RANK() OVER (PARTITION BY squad_id, position ORDER BY g_xg_diff DESC) AS squad_rank
                    FROM more_than_2
                    WHERE same_pos_squad > 2""", con=con)

,player_name,g_xg_diff,position,squad_id,squad_rank
0,Gabriel Magalhães,0.10,DF,1,1
1,Piero Hincapié,0.00,DF,1,2
2,Cristhian Mosquera,0.00,DF,1,2
3,Myles Lewis-Skelly,-0.01,DF,1,4
4,William Saliba,-0.02,DF,1,5
...,...,...,...,...,...
335,Mateus Mane,0.29,MF,20,1
336,Jean-Ricner Bellegarde,0.02,MF,20,2
337,André,-0.01,MF,20,3
338,João Gomes,-0.03,MF,20,4


In [39]:
pd.read_sql_query("""WITH players AS (
                                      SELECT stats.player_id as player_id, player_name, squad_id, positions.position AS position, 
                                      positions.pos_id, RANK() OVER (PARTITION BY positions.player_id ORDER BY pos_id) AS prim_pos, minutes
                                      FROM stats
                                      INNER JOIN positions
                                      ON stats.player_id = positions.player_id)
                    SELECT COUNT(player_id) AS num_players, position, squad_id FROM players
                    WHERE minutes > 300 AND position <> 'GK' AND prim_pos = 1
                    GROUP BY squad_id, position
                    ORDER BY num_players ASC""", con=con)

,num_players,position,squad_id
0,1,FW,8
1,1,FW,13
2,2,FW,14
3,3,FW,5
4,4,FW,4
5,4,MF,7
6,4,DF,9
7,4,MF,17
8,4,FW,19
9,5,FW,1


In [40]:
pd.read_sql_query("""WITH players AS (
                                      SELECT stats.player_id AS player_id, player_name, np_goals_per90 - npxg_per90 AS g_xg_diff, squad_id, positions.position AS position, 
                                      positions.pos_id, RANK() OVER (PARTITION BY positions.player_id ORDER BY pos_id) AS prim_pos, minutes
                                      FROM stats
                                      INNER JOIN positions
                                      ON stats.player_id = positions.player_id)
                                    SELECT player_name, g_xg_diff, position, squad_id, COUNT(player_id) OVER (PARTITION BY squad_id, position) AS same_pos_squad, minutes, prim_pos
                                    FROM players""", con=con)

,player_name,g_xg_diff,position,squad_id,same_pos_squad,minutes,prim_pos
0,Riccardo Calafiori,-0.11,DF,1,8,1255,1
1,Piero Hincapié,0.00,DF,1,8,891,1
2,Myles Lewis-Skelly,-0.01,DF,1,8,321,1
3,Gabriel Magalhães,0.10,DF,1,8,1266,1
4,Cristhian Mosquera,0.00,DF,1,8,433,1
...,...,...,...,...,...,...,...
612,Fer López,-0.07,MF,20,15,235,1
613,Mateus Mane,0.29,MF,20,15,452,1
614,Marshall Munetsi,-0.07,MF,20,15,875,1
615,David Møller Wolfe,-0.02,MF,20,15,707,2


In [49]:
pd.read_sql_query("""SELECT typeof(npxg), COUNT(typeof(npxg)) FROM stats GROUP BY typeof(npxg)""", con=con)

,typeof(npxg),COUNT(typeof(npxg))
0,real,493


In [57]:
pd.read_sql_query("""SELECT * FROM sqlite_master""", con=con)

,type,name,tbl_name,rootpage,sql
0,table,stats,stats,2,"CREATE TABLE ""stats"" (\n""index"" INTEGER,\n ""P..."
1,index,ix_stats_index,stats,3,"CREATE INDEX ""ix_stats_index""ON ""stats"" (""index"")"
2,table,positions,positions,7,"CREATE TABLE ""positions"" (\n""pos_id"" INTEGER,\..."
3,index,ix_positions_pos_id,positions,8,"CREATE INDEX ""ix_positions_pos_id""ON ""position..."


In [44]:
pd.read_sql_query("""SELECT typeof(np_goals_per90), COUNT(typeof(np_goals_per90)) FROM stats GROUP BY typeof(np_goals_per90)""", con=con)

,typeof(np_goals_per90),COUNT(typeof(np_goals_per90))
0,real,493
